In [ ]:
import json
import logging
import os
from urllib.parse import urlencode
import yaml
import pendulum
import polars as pl
import requests
import tenacity
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text
from tqdm import tqdm
import uuid
import os

In [ ]:
def load_config():
    with open("config.yaml", "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

config = load_config()

In [ ]:
formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
handler = logging.FileHandler(
    config["logging"]["file_name"],
    mode="a",
    encoding="utf8"
)
handler.setFormatter(formatter)

logging.basicConfig(level=logging.DEBUG, handlers=[handler])
logger = logging.getLogger("hh_search")

logging.getLogger("sqlalchemy.engine").setLevel(logging.INFO)
logging.getLogger("tenacity").setLevel(logging.DEBUG)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("urllib3").propagate = False

In [ ]:
def my_before_sleep(retry_state):
    logging.warning(
        f"Attempt #{retry_state.attempt_number} failed! "
        f"Error: {retry_state.outcome.exception()} "
        f"Sleeping {retry_state.upcoming_sleep} seconds"
    )

def my_before(retry_state):
    if retry_state.attempt_number >= 2:
        logging.warning(f"Starting attempt #{retry_state.attempt_number}")

def on_last_fail(retry_state):
    err = retry_state.outcome.exception()
    logging.error(
        f"Attempt #{retry_state.attempt_number} failed! "
        f"Last error: {err}"
    )
    return err

retry_http = tenacity.retry(
    stop=tenacity.stop_after_attempt(config["retry"]["attempts"]),
    wait=tenacity.wait_fixed(config["retry"]["wait_seconds"]),
    before=my_before,
    before_sleep=my_before_sleep,
    retry_error_callback=on_last_fail,
    reraise=False,
    retry=tenacity.retry_if_exception_type(
        (
            requests.RequestException,
            ValueError,
            KeyError,
            IndexError,
            json.JSONDecodeError,
        )
    ),
)

In [ ]:
def build_url(date_from: str, page: int) -> str:
    q = {
        "area": config["search"]["area"],
        "text": config["search"]["text"],
        "search_field": config["search"]["search_field"],
        "date_from": date_from,
        "page": page,
        "no_magic": str(config["search"]["no_magic"]).lower(),
        "items_on_page": config["search"]["items_on_page"],
        "order_by": config["search"]["order_by"],
        "enable_snippets": str(config["search"]["enable_snippets"]).lower(),
    }
    return f"https://hh.ru/search/vacancy?{urlencode(q)}"

In [ ]:
@retry_http
def fetch_page(url: str) -> BeautifulSoup:
    r = requests.get(
        url,
        headers={"User-Agent": config["http"]["user_agent"]},
        timeout=config["http"]["timeout"]
    )
    r.raise_for_status()
    return BeautifulSoup(r.content, "html.parser")

def parse_state(soup: BeautifulSoup) -> dict:
    tpl = soup.select_one("template#HH-Lux-InitialState")
    if tpl is None or not tpl.text.strip():
        raise ValueError("HH-Lux-InitialState template not found or empty")
    return json.loads(tpl.text)

def get_pages_count(d: dict) -> int:
    paging = d["vacancySearchResult"].get("paging")
    if paging is None:
        return 0
    if paging.get("lastPage") is None:
        return paging["pages"][-1]["page"] + 1
    return paging["lastPage"]["page"] + 1

In [ ]:
DATABASE_URL = config["database"]["url"]
db = create_engine(DATABASE_URL)

def init_db():
    with db.connect() as con:
        con.execute(text("CREATE SCHEMA IF NOT EXISTS etl;"))
        con.execute(
            text(
                """
                CREATE TABLE IF NOT EXISTS etl.hh_search (
                    created_dttm TIMESTAMPTZ NOT NULL DEFAULT NOW(),
                    etl_id UUID NOT NULL,
                    request_dttm TIMESTAMPTZ NOT NULL,
                    vacancy_id INT NOT NULL,
                    vacancy_title TEXT NOT NULL,
                    company_id INT NOT NULL,
                    company_title TEXT NOT NULL,
                    company_visible_name TEXT NOT NULL,
                    publication_time TIMESTAMPTZ NOT NULL,
                    last_change_time TIMESTAMPTZ NOT NULL,
                    creation_time TIMESTAMPTZ NOT NULL,
                    is_adv BOOL NOT NULL,
                    snippet JSONB NOT NULL,
                    responses_count INT NOT NULL,
                    total_responses_count INT NOT NULL
                )
                """
            )
        )
        con.commit()

def get_hwm() -> pendulum.DateTime:
    with db.connect() as con:
        df = pl.read_database(
            "SELECT MAX(publication_time) AS hwm FROM etl.hh_search",
            con
        )
    hwm = df["hwm"][0]
    if hwm is None:
        return pendulum.parse(config["search"]["date_from_default"])
    return pendulum.parse(str(hwm))


def parse_vacancies(d: dict, request_dttm, etl_id, hwm: pendulum.DateTime):
    vacancies = d["vacancySearchResult"]["vacancies"]
    rows = []
    for item in vacancies:
        pub_time = pendulum.parse(item["publicationTime"]["$"])
        if pub_time <= hwm:
            continue  # пропускаем старые записи

        rows.append({
            "etl_id": etl_id,
            "request_dttm": request_dttm,
            "vacancy_id": item["vacancyId"],
            "vacancy_title": item["name"],
            "company_id": item["company"]["id"],
            "company_title": item["company"]["name"],
            "company_visible_name": item["company"]["visibleName"],
            "publication_time": pub_time,
            "last_change_time": pendulum.parse(item["lastChangeTime"]["$"]),
            "creation_time": pendulum.parse(item["creationTime"]),
            "is_adv": item.get("@isAdv", "false") in (True, "true", "True", "1"),
            "snippet": json.dumps(item.get("snippet", {}), ensure_ascii=False),
            "responses_count": item["responsesCount"],
            "total_responses_count": item["totalResponsesCount"],
        })
    return rows

In [11]:
def main():
    etl_id = str(uuid.uuid4())
    try:
        logger.info(f"ETL started | etl_id={etl_id}")
        init_db()

        hwm = get_hwm()
        logger.info(f"etl_id={etl_id} | HWM: {hwm}")

        first_url = build_url(date_from=str(hwm), page=0)
        soup = fetch_page(first_url)
        d = parse_state(soup)
        pages_count = get_pages_count(d)

        if pages_count == 0:
            logger.info(f"etl_id={etl_id} | Нет новых вакансий")
            return

        logger.info(f"etl_id={etl_id} | pages_count: {pages_count}")

        all_new_rows = []

        for page_idx in tqdm(range(pages_count)):
            logger.info(f"etl_id={etl_id} | page_idx={page_idx}")

            url = build_url(date_from=str(hwm), page=page_idx)
            request_time = pendulum.now("UTC")

            try:
                soup = fetch_page(url)
                d = parse_state(soup)
                rows = parse_vacancies(d, request_time, etl_id, hwm)

                if not rows:
                    logger.info(f"etl_id={etl_id} | page_idx={page_idx} | новых вакансий нет")
                    continue

                df = pl.DataFrame(rows)
                df.write_database("etl.hh_search", db, if_table_exists="append")
                logger.info(f"etl_id={etl_id} | page_idx={page_idx} | inserted {len(rows)} rows")

                all_new_rows.extend(rows)

            except Exception as e:
                logger.exception(f"etl_id={etl_id} | page_idx={page_idx} | failed: {e}")
                raise

        if all_new_rows:
            df_all = pl.DataFrame(all_new_rows)
            output_dir = "data"
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"hh_search_{etl_id}.parquet")
            df_all.write_parquet(output_path, compression="snappy")
            logger.info(f"etl_id={etl_id} | saved {len(all_new_rows)} new rows to {output_path}")

        logger.info(f"ETL finished | etl_id={etl_id}")

    except Exception as e:
        logger.exception(f"etl_id={etl_id} | Fatal error: {e}")
        raise

if __name__ == "__main__":
    main()

100%|██████████████████████████████████████████████████████████████████████████████████| 33/33 [00:57<00:00,  1.74s/it]
